# Tutorial for the Sparse Szegedy Simulator Module

Sergio A. Ortega and Daniel K. Park

In this notebook we show a tutorial of how to use the sparse framework of our package SQWLib [[1]](...) for simulating the Szegedy quantum walk [[2]](https://ieeexplore.ieee.org/document/1366222) on sparse graphs. This library follows a similiar function to SQUWALS for densde graphs [[3]](https://advanced.onlinelibrary.wiley.com/doi/full/10.1002/qute.202400022), once defined the state space data. See: https://github.com/OrtegaSA/squwals-repo/blob/main/Tutorials/SQUWALS%20Tutorial.ipynb.

It is recommended to read section II from [[1]](https://...) for terminology.

## Table of Contents

- [Packages](#0)
- [1 - Space Data and Transition Matrix](#1)
- [2 - Initial State](#2)
- [3 - Unitary Operators](#3)
    - [3.1 - Reflection](#3-1)
    - [3.2 - Swap](#3-2)
    - [3.3 - Oracle](#3-3)
    - [3.4 - Update and Reflection Around 0](#3-4)
    - [3.5 - Unitary Operator](#3-5)
- [4 - Measurement](#4)
- [5 - Szegedy Quantum Walk Simulator](#5)
- [6 - Sparse and Dense Interconversions](#6)
- [7 - References](#7)

<a name='0'></a>
## Packages
First we import the libraries that we need, including our simulator.

In [1]:
import sys
sys.path.insert(0, '../')

In [2]:
import numpy as np
import matplotlib.pyplot as plt

import sqwlib as sqw

<a name='1'></a>
## 1 - Space Data and Transition Matrix
First, we need the transition matrix of a classical walk $G$, whose elements $G_{ji}$ are the probabilities of the walker jumping from node $i$ to node $j$. This is a $N\times N$ column-stochastic matrix, so that the elements of each column add up to one. Thus:

$$\sum_{j=0}^{N-1} G_{ji} = 1 \ \forall \ i.$$

**Note: Depending on the literature, the transition matrix is defined as row-stochastic, so that it corresponds to our $G^T$. For this simulator we ALWAYS need a column-stochastic matrix.**

For the sparse modulus, we only store the non-null weights of the graphs. Thus, we need a list with the weights associated to the edges of the graph, so that the elements are $(i,j,G_{ji})$.

Let us create an example for a graph with $N=4$ nodes.

In [3]:
N = 4

weights_list = [
    (0,1,0.5), # Probability 0.5 from node 0 to node 1.
    (0,2,0.5),
    (1,0,0.2),
    (1,1,0.2),
    (1,3,0.6),
    (2,0,0.2),
    (2,3,0.8),
    (3,1,0.5),
    (3,3,0.5),
    ]

We use this list to create an instance of the `SpaceData` class. The parameters are the following:
- edge_list: List of edges of the graph.
- add_zeroes: If True, edges connected to node 0 are added. Default: False.

Actually, the list does not need to contain the weights, but only the edges. However, we usually are given the graph along the weights, so that by convenience we directly store the `weights_list` like this.

Regarding the edges connected to node $0$, they must be added in case of using the update $V$ and the reflection $R_0$ operators, which need those edge states in the Hilbert space.

In [4]:
sd = sqw.SpaceData(weights_list)

Once defined the space data, we use it to obtain the sparse transition matrix as a numpy vector applying the method `obtain_transition_matrix` to the list of weights. 

In [5]:
transition_matrix = sd.obtain_transition_matrix(weights_list)

We can check if the created transition matrix is column-stochastic with the method `check_transition_matrix`.

In [6]:
sd.check_transition_matrix(transition_matrix)

True

If it would not be column-stochastic, we could normalize it with the method `normalize_transition_matrix`.

In [7]:
normalized_transition_matrix = sd.normalize_transition_matrix(transition_matrix)
sd.check_transition_matrix(normalized_transition_matrix)

True

From this point, the `SpaceData` object has the same methods as the package SQUWALS. Thus, the rest of codes are similiar to those of the library SQUWALS, making the following change:

`sw.function(input) --> sd.function(input)`

<a name='2'></a>
## 2 - Initial State

The Hilbert space is the span of all the vectors representing the $N \times N$ directed edges of the duplicated graph, i.e., $\mathcal{H} = \text{span}\lbrace\left|i\right>_1\left|j\right>_2,\ i,j = 0,1,...,N-1\rbrace = \mathbb{C}^N \otimes \mathbb{C}^N$, where the states with indexes $1$ and $2$ refer to the nodes on two copies of the original graph. Thus, the states are defined on two quantum registers. Therefore, any quantum state is represented by a $N^2$-dimensional vector.

We need an initial quantum state over which to perform the quantum evolution. An usual initial state is defined as

\begin{equation}
	\left|\Psi^{(0)}\right> = \sum_{i=0}^{N-1}c_i\left|\psi_i\right>,
\end{equation}

where the states

\begin{equation}\label{psi_i}
	\left|\psi_i\right> := \left|i\right>_1 \otimes \sum_{k=0}^{N-1} \sqrt{G_{ki}}\left|k\right>_2
\end{equation}
are created using the transition matrix $G$.

We provide the method `create_initial_state` to create such initial state with a sparse representation. The arguments are the following:
- transition_matrix: Sparse column-stochastic transition matrix.
- coefficients: List of coefficients for the linear combination of the psi_i states. Default: 1/np.sqrt(len(nodes)).
- nodes: List of nodes corresponding to the psi_i states of the linear combination. Default: all nodes.

If only the transition matrix is provided, it creates the uniform superposition of all the states.

In [8]:
initial_state = sd.create_initial_state(transition_matrix)

If we pass a list of nodes, it creates the uniform superposition for those nodes.

In [9]:
nodes = [1,2]
initial_state = sd.create_initial_state(transition_matrix,nodes=nodes)

Finally, we can also pass an array with the complex amplitudes $c_i$ for the linear combination.

In [10]:
nodes = [0,1]
coefficients = [np.sqrt(0.3),np.sqrt(0.7)]
initial_state = sd.create_initial_state(transition_matrix,nodes=nodes,coefficients=coefficients)

So far, we have created an initial state in the usual way, depending on the transition matrix weights. However, for alternative formulations, an initial state independent of the transition matrix can be used. This is defined as

\begin{equation}
	\left|\Psi^{(0)}\right> = \sum_{i=0}^{N-1}c_i\left|i\right>_1\left|0\right>_2.
\end{equation}

To create it, we can use the method `create_no_coin_state`, whose areguments are:

- N: Number of nodes of the graph.
- coefficients: List of coefficients for the linear combination of the psi_i states. Default: 1/np.sqrt(len(nodes)).
- nodes: List of nodes corresponding to the psi_i states of the linear combination. Default: all nodes.

**Note: In this case we need the edge states connecting with node $0$.**

In [11]:
sd = sqw.SpaceData(weights_list,add_zeroes=True)

nodes = [0,1]
coefficients = [np.sqrt(0.3),np.sqrt(0.7)]
initial_state = sd.create_no_coin_state(N,nodes=nodes,coefficients=coefficients)

<a name='3'></a>
## 3 - Unitary operators

The quantum walk operator $U$ is usually defined as

\begin{equation}\label{U}
	U := SR,
\end{equation}

where $R$ is a reflection over the subspace generated by the $\left|\psi_i\right>$ states,

\begin{equation}\label{reflection}
R = 2\Pi - \mathbb{1}, \ \ \ \ \ \ \Pi := \sum_{i=0}^{N-1} \left|\psi_i\right>\left<\psi_i\right|,
\end{equation}

and $S$ is the swap operator between the two quantum registers, i.e.,

\begin{equation}\label{swap}
	S := \sum_{i,j=0}^{N-1} \left|i,j\right>\left<j,i\right|.
\end{equation}

However, there exists also the double Szegedy operator $U = SRSR$.

**Note: The original formulation of the unitary is as two reflections, $U = R_BR_A$, where $R_A = R$ and $R_B = SRS$. Thus, it corresponds to the double Szegedy operator in our notation.**

Furthermore, the unitary operator can be modified introducing oracles in the context of quantum search. Let us define the operator $Q$ as follows:

\begin{equation}\label{oracle}
Q = \mathbb{1}_N - 2\sum_{k \in \mathcal{M}} \left|k\right>\left<k\right|,
\end{equation}

which inverts the sign of the vectors $\left|k\right>$ given a set $\mathcal{M}$ of nodes to mark. In order to mark the nodes in the first register, we use the oracle $Q_1$ defined as:

\begin{equation}
Q_1 = Q \otimes \mathbb{1}_N,
\end{equation}

whereas to mark them in the second register, we use $Q_2$:

\begin{equation}
Q_2 = \mathbb{1}_N \otimes Q.
\end{equation}

Finally, different fomrulations of the walk can be done in terms of two more fundamental operators, which are the update oprator $V$ and the reflection $R_0$ around the states with $0$ in the second register. These operators can produce the original reflection $R$ since:

\begin{equation}
	V\left|i\right>_1\left|0\right>_2 = \left|\psi_i\right>,
\end{equation}

\begin{equation}
	R_0 = 2\sum_{i=0}^{N-1} \left|i,0\right>\left<i,0\right| - \mathbb{1},
\end{equation}

\begin{equation}
	R = V R_0 V^\dagger.
\end{equation}

Any Szegedy-like quantum operator can be expressed in terms of these opeartors. Thus, we will see how to create these operators, and use them to build the unitary quantum evolution $U$.

<a name='3-1'></a>
### 3.1 - Reflection

We provide the `Reflection` class as a method of the `SpaceData` object. To instantiate a object we need to pass as argument the transition matrix $G$.

In [12]:
sd = sqw.SpaceData(weights_list)
transition_matrix = sd.obtain_transition_matrix(weights_list)

R = sd.Reflection(transition_matrix)
R.info()

Sparse Reflection


<a name='3-2'></a>
### 3.2 - Swap

We provide the `Swap` class as a method of the `SpaceData` object. No arguments are needed.

In [13]:
S = sd.Swap()
S.info()

Sparse Swap


<a name='3-3'></a>
### 3.3 - Oracle

We provide the `Oracle` class as a method of the `SpaceData` object. The arguments are the following:

- register: An intenger indicating the register for marking: 1 or 2.
- marked_nodes: A list with the nodes to mark.

Let us create an oracle that marks nodes 0 and 2 in the first register.

In [14]:
Q1 = sd.Oracle(1,marked_nodes=[0,2])
Q1.info()

Sparse Oracle: Register 1: nodes [0, 2]


Now for the second register.

In [15]:
Q2 = sd.Oracle(2,marked_nodes=[0,2])
Q2.info()

Sparse Oracle: Register 2: nodes [0, 2]


<a name='3-4'></a>
### 3.4 - Update and Reflection Around 0

For using these operators, we again need the edges connecting to node $0$.

In [16]:
sd = sqw.SpaceData(weights_list,add_zeroes=True)
transition_matrix = sd.obtain_transition_matrix(weights_list)

In the case of the update operator $V$, it can be defined in several ways. In our simulator, we provide to methods for simulating the update operator.

- Method 1:

\begin{equation}
	V = \sum_{i=0}^{N-1} \left|\psi_i\right>\left<i,0\right| + \sum_{i=0}^{N-1} \left|\psi_i^\perp\right>\left<i,0^\perp\right| + \mathbb{1} - \sum_{i=0}^{N-1} \left|i,0\right>\left<i,0\right| - \sum_{i=0}^{N-1} \left|i,0^\perp\right>\left<i,0^\perp\right|.
\end{equation}

- Method 2:

\begin{equation}
	V = \mathbb{1} - 2\sum_{i=0}^{N-1}\left|i,v_i\right>\left<i,v_i\right|.
\end{equation}

We provide the `Update` class as a method of the `SpaceData`object. To instantiate a object we need to pass as argument the transition matrix $G$, as well as the method number.

In [17]:
V = sd.Update(transition_matrix) # By default method = 2.
# V = sd.Update(transition_matrix, method = 1)
# V = sd.Update(transition_matrix, method = 2)

V.info()

Sparse Update


For the inverse of the update operator $V^\dagger$ we provide the `UpdateDagger` class as a method of the `SpaceData`object. However, it can also be crated using the `inverse` method of the `Update` object.

In [18]:
Vd = sd.UpdateDagger(transition_matrix)
# Vd = sd.UpdateDagger(transition_matrix, method = 1)
# Vd = sd.UpdateDagger(transition_matrix, method = 2)

Vd.info()

Sparse Update dagger


In [19]:
Vd = V.inverse()
Vd.info()

Update dagger


Finally, for the reflection $R_0$ we provide the `Reflection0` class as a method of the `SpaceData`object. No arguments are needed.

In [20]:
R0 = sd.Reflection0()
R0.info()

Sparse Reflection 0


<a name='3-5'></a>
### 3.5 - Unitary Operator

Now that we have seen how to create the building blocks, we can join them into an unitary operator $U$. Let us create the operator

$$U = V^\dagger SVR_0V^\dagger SVR_0.$$

Note that the order of action of the operators is read from right to left. Thus, the first operator is the reflection $R_0$.

We can create an unitary object with the `Unitary` class as a method of the `SpaceData`object, and use the `append` method to introduce the building blocks.

In [21]:
sd = sqw.SpaceData(weights_list,add_zeroes=True)
transition_matrix = sd.obtain_transition_matrix(weights_list)
V = sd.Update(transition_matrix)
Vd = V.inverse()
R0 = sd.Reflection0()
S = sd.Swap()

U = sd.Unitary()

# First we add the reflection around 0.
U.append(R0)

# Secondly the update oeprator.
U.append(V)

# And so on
U.append(S)
U.append(Vd)
U.append(R0)
U.append(V)
U.append(S)
U.append(Vd)

print(U)
U.info()

V†SVR₀V†SVR₀
Custom sparse unitary:
 1 - Sparse Reflection 0
 2 - Sparse Update
 3 - Sparse Swap
 4 - Update dagger
 5 - Sparse Reflection 0
 6 - Sparse Update
 7 - Sparse Swap
 8 - Update dagger


Another form of initializing the unitary is passing a list with the operators. The first operator of the list is the first one acting.

In [22]:
U = sd.Unitary(operators=[R0,V,S,Vd,R,V,S,Vd])
print(U)
U.info()

V†SVRV†SVR₀
Custom sparse unitary:
 1 - Sparse Reflection 0
 2 - Sparse Update
 3 - Sparse Swap
 4 - Update dagger
 5 - Sparse Reflection
 6 - Sparse Update
 7 - Sparse Swap
 8 - Update dagger


Finally, other form of constructing the unitary operator is multiplying the different operators in an algebraic sense, using `*`.

In [23]:
U = Vd * S * V * R0 * Vd * S * V * R0
print(U)
U.info()

V†SVR₀V†SVR₀
Custom sparse unitary:
 1 - Sparse Reflection 0
 2 - Sparse Update
 3 - Sparse Swap
 4 - Update dagger
 5 - Sparse Reflection 0
 6 - Sparse Update
 7 - Sparse Swap
 8 - Update dagger


Finally, we can use the method `operate` passing a quantum state, returning the action of the unitary on it. The initial vector can be a 1D numpy array.

In [24]:
final_state = U.operate(initial_state)

However, we can concatenate different initial states along a second dimension. Let us create a batch with two quantum states.

In [25]:
initial_batch = np.c_[initial_state,initial_state]
initial_batch

array([[0.54772256, 0.54772256],
       [0.        , 0.        ],
       [0.        , 0.        ],
       [0.        , 0.        ],
       [0.83666003, 0.83666003],
       [0.        , 0.        ],
       [0.        , 0.        ],
       [0.        , 0.        ],
       [0.        , 0.        ],
       [0.        , 0.        ],
       [0.        , 0.        ],
       [0.        , 0.        ],
       [0.        , 0.        ]])

When we apply the unitary operator over a batch of states, we obtain a batch with the same dimensions.

In [26]:
final_batch = U.operate(initial_batch)
final_batch

array([[-2.22803482e-01, -2.22803482e-01],
       [ 7.48331477e-02,  7.48331477e-02],
       [-7.48331477e-02, -7.48331477e-02],
       [ 1.11022302e-16,  1.11022302e-16],
       [-3.11171709e-02, -3.11171709e-02],
       [-4.44057893e-02, -4.44057893e-02],
       [ 4.36032014e-02,  4.36032014e-02],
       [ 1.67332005e-01,  1.67332005e-01],
       [ 3.34664011e-01,  3.34664011e-01],
       [ 8.31297257e-01,  8.31297257e-01],
       [-6.02581423e-02, -6.02581423e-02],
       [ 3.09838668e-01,  3.09838668e-01],
       [ 6.02581423e-02,  6.02581423e-02]])

<a name='4'></a>
## 4 - Measurement

In order to obtain a probability distribution for the position of the walker, we can use the `Measurement` class as a method of the `SpaceData`object.

The only argument for initializing this class is the register used to measure the position. Let us create an object for the first register.

In [27]:
measurement = sd.Measurement(1)

Let us obtain the probability distribution of the walker position after measuring the first register for the 1D array state. We have to use the `operate` method. In this case, the result will be a 1D array too, with $N$ elements.

In [28]:
probability_distribution = measurement.operate(final_state)
probability_distribution

array([0.06084139, 0.00484139, 0.14      , 0.79431722])

We can also measure a batch of states, resulting in a 2D array where the first dimension indicates the node and the second dimension the quantum state.

In [29]:
probability_batch = measurement.operate(final_batch)
probability_batch

array([[0.06084139, 0.06084139],
       [0.00484139, 0.00484139],
       [0.14      , 0.14      ],
       [0.79431722, 0.79431722]])

<a name='5'></a>
## 5 - Szegedy Quantum Walk Simulator

With the unitary and measurement classes we can simulate the Szegedy quantum walk from any initial quantum state. However, we also provide a higher-level simulator that makes the simulation for a given number of steps, retrieving the probability distribution of the walker at each step.

The function `quantum_szegedy_simulator` is provided as a method of the `SpaceData`class, and has the following arguments:

- unitary: unitary operator model.
- initial_state: Initial quantum state, or batch of states.
- time_steps: Number of steps of the quantum walk.
- measure: An intenger indicating the register for measuring.

This function returns directly a tensor with all the probability distributions, where the first index inidicates the time step (including $t = 0$).

Let us see an example using the previous initial state, simulating for 20 time steps with the operator $U$ and measuring the first register.

In [30]:
probability_distribution_times = sd.quantum_szegedy_simulator(unitary=U,initial_state=initial_state,time_steps=20,measure=1)
probability_distribution_times.shape

(21, 4)

We have 21 distributions because the first one is at time 0, and each of them has 4 elements, which are the probability for each of the nodes.

We can take the probability distribution at any time. For example, for $t=1$, we must obtain the same result than the simulations of the previous section where we applied $U$ only once.

In [31]:
probability_distribution_times[1]

array([0.06084139, 0.00484139, 0.14      , 0.79431722])

<a name='6'></a>
## 6 - Sparse and Dense Interconversions

Given a quantum state in sparse form, we can convert it to its dense representation using the method `to_dense_state`of the SpaceData object.

In [32]:
dense_state = sd.to_dense_state(initial_state)
dense_state

array([0.54772256, 0.        , 0.        , 0.        , 0.83666003,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        ])

Moreover, we can also transform the sparse transition matrix in its dense representation with the method `to_dense_matrix`.

In [33]:
dense_transition_matrix = sd.to_dense_matrix(transition_matrix)
dense_transition_matrix

array([[0. , 0.2, 0.2, 0. ],
       [0.5, 0.2, 0. , 0.5],
       [0.5, 0. , 0. , 0. ],
       [0. , 0.6, 0.8, 0.5]])

For going from the dense representation to the sparse representation we use the methods `to_sparse_state` and `to_sparse_matrix`.

In [34]:
sparse_state = sd.to_sparse_state(dense_state)
sparse_state

array([0.54772256, 0.        , 0.        , 0.        , 0.83666003,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        ])

In [35]:
sparse_transition_matrix = sd.to_sparse_matrix(dense_transition_matrix)
sparse_transition_matrix

array([0. , 0.5, 0.5, 0. , 0.2, 0.2, 0.6, 0.2, 0.8, 0. , 0.5, 0. , 0.5])

<a name='7'></a>
## 7 - References

- [1] **[S. A. Ortega and D. K. Park. Paper Reference.](...)**
- [2] **[M. Szegedy. Quantum speed-up of Markov chain based algorithms. 45th Annual IEEE Symposium on Foundations of Computer Science, 32–41, 2004.](https://ieeexplore.ieee.org/document/1366222)**
- [3] **[S. A. Ortega and M. A. Martin-Delgado. SQUWALS: A Szegedy QUantum WALks Simulator. Advanced Quantum Technologies, 7:2400022, 2024.](https://advanced.onlinelibrary.wiley.com/doi/full/10.1002/qute.202400022)**